In [1]:
import pandas as pd
from transformers import pipeline
from keybert import KeyBERT
from tqdm import tqdm
import os

# Файлы
INPUT_FILE = "mit_oai_all.csv"
OUTPUT_FILE = "mit_oai_with_ner_keywords.csv"
TEMP_FILE = "mit_oai_with_ner_keywords_temp.csv"

# Загружаем исходные данные
df = pd.read_csv(INPUT_FILE)

# Если есть временный файл, продолжаем с последней строки
start_idx = 0
if os.path.exists(TEMP_FILE):
    temp_df = pd.read_csv(TEMP_FILE)
    start_idx = len(temp_df)
    print(f"Продолжаем с строки {start_idx}")

# NER модель
ner = pipeline(
    "ner",
    model="tner/deberta-v3-large-ontonotes5",
    aggregation_strategy="simple",
    tokenizer="tner/deberta-v3-large-ontonotes5",
    device=0
)

# Модель ключевых слов
kw_model = KeyBERT(model="all-mpnet-base-v2")

# Функции
def extract_ner(text, max_len=3000):
    if not isinstance(text, str) or not text.strip():
        return []
    text = text[:max_len]
    try:
        ents = ner(text)
        return [
            {"word": e["word"], "label": e["entity_group"], "score": round(e["score"], 4)}
            for e in ents
        ]
    except:
        return []

def extract_keywords(text, top_n=10):
    if not isinstance(text, str) or not text.strip():
        return []
    try:
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 3),
            stop_words="english",
            use_mmr=True,
            diversity=0.7,
            top_n=top_n
        )
        return [
            {"keyword": kw, "score": round(float(score), 4)}
            for kw, score in keywords
        ]
    except:
        return []

# Обработка строк
ner_results = []
kw_results = []

# Загружаем уже обработанные строки, если есть
if os.path.exists(TEMP_FILE):
    temp_df = pd.read_csv(TEMP_FILE)
    ner_results = temp_df["ner_entities"].tolist()
    kw_results = temp_df["keywords"].tolist()

# Обрабатываем новые строки
for idx in tqdm(range(start_idx, len(df))):
    row = df.iloc[idx]
    text = " ".join([
        str(row.get("title", "")),
        str(row.get("description", ""))
    ])

    # NER
    ner_out = extract_ner(text)
    ner_str = "; ".join([f"{e['word']}|{e['label']}|{e['score']}" for e in ner_out])

    # Keywords
    kw_out = extract_keywords(text)
    kw_str = "; ".join([f"{k['keyword']}|{k['score']}" for k in kw_out])

    ner_results.append(ner_str)
    kw_results.append(kw_str)

    # Периодическое сохранение каждые 100 строк
    if idx % 100 == 0:
        pd.DataFrame({
            "ner_entities": ner_results,
            "keywords": kw_results
        }).to_csv(TEMP_FILE, index=False)
        print(f"Промежуточное сохранение: {TEMP_FILE} (строка {idx})")

# Сохраняем финальные результаты
result_df = df.copy()
result_df["ner_entities"] = ner_results
result_df["keywords"] = kw_results
result_df.to_csv(OUTPUT_FILE, index=False)

# Удаляем временный файл
if os.path.exists(TEMP_FILE):
    os.remove(TEMP_FILE)

print("Готово:", OUTPUT_FILE)

d:\MIT-collect\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Продолжаем с строки 121801


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 30804.07it/s]
DebertaV2ForTokenClassification LOAD REPORT from: tner/deberta-v3-large-ontonotes5
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5516.85it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  0%|          | 100/29672 [00:27<3:13:02,  2.55it/s]

Промежуточное сохранение: mit_oai_with_ner_keywords_temp.csv (строка 121900)


  1%|          | 201/29672 [00:52<4:15:37,  1.92it/s]

...

Промежуточное сохранение: mit_oai_with_ner_keywords_temp.csv (строка 151300)


100%|█████████▉| 29600/29672 [8:04:15<01:44,  1.45s/it]

Промежуточное сохранение: mit_oai_with_ner_keywords_temp.csv (строка 151400)


100%|██████████| 29672/29672 [8:05:34<00:00,  1.02it/s]


Готово: mit_oai_with_ner_keywords.csv
